## Step 2.1: Hardware Foundations (CPU, Cores, & Registers)
**First Principle:** A Central Processing Unit (CPU) is an electronic state machine driven by an oscillator clock. At physical layer zero, a single CPU core can only decode and execute a single stream of instructions sequentially at any single clock cycle.

**Analogy:** Think of a CPU core as a single mathematician with a scratchpad (Registers) and a very small desk (L1/L2 Cache). No matter how fast they write, they can only solve one equation at any given instant. A multi-core CPU is simply multiple mathematicians sitting in the same office building, each with their own desk and scratchpad.

```text
+-------------------------------------------------------------------+
|                        PHYSICAL CPU SOCKET                        |
|                                                                   |
|  +-----------------------------+   +---------------------------+  |
|  |           CORE 0            |   |          CORE 1           |  |
|  |  +-----------------------+  |   |  +---------------------+  |  |
|  |  | Registers & ALU       |  |   |  | Registers & ALU     |  |  |
|  |  +-----------------------+  |   |  +---------------------+  |  |
|  |  | L1 / L2 Cache Memory  |  |   |  | L1 / L2 Cache Memory|  |  |
|  +-----------------------------+   +---------------------------+  |
|                                                                   |
|  +-------------------------------------------------------------+  |
|  |                   Shared L3 Cache Memory                    |  |
|  +-------------------------------------------------------------+  |
+-------------------------------------------------------------------+
                                  |
                                  v
                     [ Physical RAM / Main Memory ]

1. **Registers & ALU:** The Arithmetic Logic Unit (ALU) performs raw binary math; registers hold the immediate inputs and outputs within sub-nanosecond access times.

2. **Hardware Core:** An independent physical processing unit containing its own execution pipeline and local caches.

3. **Hyper-Threading (Logical Cores):** Duplicates the architectural state (registers, program counter) on a single physical core so it can interleave two instruction streams when one is stalled waiting for memory.

In [3]:
import os
import multiprocessing

# Inspect physical vs logical execution units available to the OS
logical_cores = os.cpu_count()
print(f"Total Logical CPU Execution Units (Cores/Threads): {logical_cores}")

# Display CPU affinity (which logical cores this Python process is allowed to run on)
try:
    affinity = os.sched_getaffinity(0)
    print(f"Current Process CPU Core Affinity: {affinity}")
except AttributeError:
    # os.sched_getaffinity is Unix-specific
    print("Affinity query not supported on this OS platform.")

Total Logical CPU Execution Units (Cores/Threads): 8
Affinity query not supported on this OS platform.


## Step 2.2: OS Abstractions (Process vs. Thread)
**First Principle:** The Operating System (OS) isolates applications using virtual memory address spaces. A **Process** is an instance of a computer program allocated its own isolated memory space. A **Thread** is the smallest unit of execution *within* that process, sharing the parent process's memory space (Heap), but retaining its own execution stack and register state.

**Analogy:** 
* **Process:** An entire independent workshop building. It has its own private tool supply and storage room[cite: 5]. One workshop cannot accidentally break or access items in another workshop.
* **Thread:** Multiple workers inside the *same* workshop building. They can talk to each other instantly and share tools directly (Shared Heap), but if one worker spills paint on the floor, all workers are affected (Race conditions / Crashes).

```text
======================= OS MEMORY ARCHITECTURE =======================

+-------------------------------------------------------------------+
| PROCESS A (PID: 1001) - Isolated Virtual Address Space            |
|                                                                   |
|  +-------------------------------------------------------------+  |
|  | SHARED HEAP (Global variables, Allocated Objects, Code)     |  |
|  +-------------------------------------------------------------+  |
|                                                                   |
|  +---------------------------+     +---------------------------+  |
|  | THREAD 1                  |     | THREAD 2                  |  |
|  | - Program Counter (PC)    |     | - Program Counter (PC)    |  |
|  | - Private Call Stack      |     | - Private Call Stack      |  |
|  | - Register Snapshot       |     | - Register Snapshot       |  |
|  +---------------------------+     +---------------------------+  |
+-------------------------------------------------------------------+
                                  X  (Protected boundary: No direct access)
+-------------------------------------------------------------------+
| PROCESS B (PID: 1002) - Isolated Virtual Address Space            |
|  +-------------------------------------------------------------+  |
|  | SHARED HEAP                                                 |  |
|  +-------------------------------------------------------------+  |
+-------------------------------------------------------------------+

1. **Context Switch:** When the OS pauses one thread/process to run another, saving CPU registers to RAM and restoring the next thread's state. Process context switches are computationally expensive due to page table and Translation Lookaside Buffer (TLB) flushes; thread context switches inside the same process are faster.

2. **Concurrency vs. Parallelism:** Concurrency is dealing with multiple things at once (interleaving execution on 1 core). Parallelism is doing multiple things at the exact same physical instant (executing on $\ge 2$ physical cores).

In [4]:
import os
import threading
import multiprocessing

# Verify memory isolation between Processes vs shared memory in Threads
shared_state = 0

def thread_worker():
    global shared_state
    shared_state += 10
    print(f"[Thread] PID: {os.getpid()} | Mutated shared_state -> {shared_state}")

def process_worker():
    global shared_state
    shared_state += 50
    print(f"[Process] PID: {os.getpid()} | Mutated isolated shared_state -> {shared_state}")

print(f"[Main Baseline] PID: {os.getpid()} | Initial shared_state: {shared_state}")

# 1. Run a Thread (Shares memory space)
t = threading.Thread(target=thread_worker)
t.start()
t.join()
print(f"[Main After Thread] shared_state is now: {shared_state}")

# 2. Run a Process (Copies/isolates memory space)
p = multiprocessing.Process(target=process_worker)
p.start()
p.join()
print(f"[Main After Process] shared_state is still: {shared_state} (Isolated memory!)")

[Main Baseline] PID: 25448 | Initial shared_state: 0
[Thread] PID: 25448 | Mutated shared_state -> 10
[Main After Thread] shared_state is now: 10
[Main After Process] shared_state is still: 10 (Isolated memory!)
